# CLM Conversion Kill Test 001

Formal hosted-GPU execution for the frozen Mature MoE Functional Cellization protocol. Each terminal seed is committed and pushed immediately.

In [ ]:
from pathlib import Path
import json, os, subprocess, sys

ROOT = Path('/kaggle/working/mini-cells')
BRANCH = 'codex/clm-conversion-kill-test-001'
if not ROOT.exists():
    subprocess.run(['git', 'clone', 'https://github.com/ArcheLabs/mini-cells.git', str(ROOT)], check=True)
subprocess.run(['git', 'fetch', 'origin', BRANCH], cwd=ROOT, check=True)
subprocess.run(['git', 'checkout', '-B', BRANCH, f'origin/{BRANCH}'], cwd=ROOT, check=True)
print({'head': subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=ROOT, text=True).strip()})

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'transformers==5.0.0', 'huggingface_hub==1.11.0', 'safetensors==0.7.0'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[lm,dev]'], cwd=ROOT, check=True)
subprocess.run([sys.executable, '-m', 'pytest', '-q', 'tests/test_functional_cellization.py', 'tests/test_clm_conversion_kill_test_001.py'], cwd=ROOT, check=True)

In [ ]:
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    os.environ['GITHUB_TOKEN'] = secrets.get_secret('GITHUB_TOKEN')
    try:
        os.environ['HF_TOKEN'] = secrets.get_secret('HF_TOKEN')
    except Exception:
        pass
except Exception as exc:
    raise RuntimeError('Kaggle Secret GITHUB_TOKEN is required') from exc

subprocess.run([
    sys.executable, 'scripts/research/clm_conversion_kill_test_001/publish.py',
    '--branch', BRANCH, '--preflight-only'
], cwd=ROOT, check=True)

In [ ]:
import torch, transformers, huggingface_hub
assert torch.cuda.is_available(), 'CUDA is required for formal execution'
protocol = json.loads((ROOT / 'research/validations/clm-conversion-kill-test-001/protocol.json').read_text())
free_mb, total_mb = torch.cuda.mem_get_info(0)
free_mb //= 1024 * 1024
total_mb //= 1024 * 1024
assert free_mb >= 12000, f'need >= 12000 MiB free GPU memory, found {free_mb}'
print({
    'gpu': torch.cuda.get_device_name(0),
    'free_mb': free_mb,
    'total_mb': total_mb,
    'torch': torch.__version__,
    'transformers': transformers.__version__,
    'huggingface_hub': huggingface_hub.__version__,
    'formal_seeds': protocol['formal_seeds'],
    'foundation': protocol['base']['model_id'],
    'revision': protocol['base']['revision'],
    'layers': protocol['substrate']['layer_indices'],
})

In [ ]:
def run_compact(command, log_path):
    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    with log_path.open('w', encoding='utf-8') as handle:
        process = subprocess.Popen(command, cwd=ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=os.environ.copy())
        assert process.stdout is not None
        for line in process.stdout:
            handle.write(line)
            handle.flush()
            if line.startswith('[conversion001]'):
                print(line, end='')
        returncode = process.wait()
    if returncode != 0:
        tail = log_path.read_text(encoding='utf-8', errors='replace').splitlines()[-100:]
        print('\n=== Child log tail ===')
        print('\n'.join(tail))
        subprocess.run(['nvidia-smi'], check=False)
        raise RuntimeError(f'formal child failed with exit code {returncode}; full log: {log_path}')

artifact_root = ROOT / 'artifacts/experiments/clm-conversion-kill-test-001'
for seed in protocol['formal_seeds']:
    durable = artifact_root / f'seed-{seed}/seed_summary.json'
    if durable.is_file():
        print(f'[conversion001][seed={seed}] already published; skipping')
        continue
    run_compact([
        sys.executable, 'scripts/research/clm_conversion_kill_test_001/run_seed.py',
        '--seed', str(seed), '--device', 'cuda:0'
    ], ROOT / f'results/clm-conversion-kill-test-001-launcher/seed-{seed}.log')
    subprocess.run([
        sys.executable, 'scripts/research/clm_conversion_kill_test_001/publish.py',
        '--seed', str(seed), '--branch', BRANCH
    ], cwd=ROOT, check=True)
    decision = json.loads((artifact_root / 'decision.json').read_text())
    print({
        'status': decision['status'],
        'completed_seeds': decision['completed_seeds'],
        'passed_seeds': decision['passed_seeds'],
    })

In [ ]:
decision_path = ROOT / 'artifacts/experiments/clm-conversion-kill-test-001/decision.json'
if decision_path.is_file():
    decision = json.loads(decision_path.read_text())
    print(json.dumps(decision, indent=2, sort_keys=True))

Recovery rule: rerun the notebook. Any completed PASS or FAIL seed already committed under `artifacts/experiments/clm-conversion-kill-test-001/` is skipped. Scientific thresholds remain frozen.